# Mixture of Scars (MoS) 32B (Layer 32) + SCache Tri-Compass Ensemble

Ultra-fast, lean architecture removing 56 GB 14B dependencies. Ensemble: LinearSVC (1.0) + DeBERTa-v3-large (0.35) + MoS 32B Layer 32 Tri-Compass (0.85 + 0.25 twin) + Soft Sinkhorn.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights. Every member adds weight * log(probability) to the score.
W_SVC = 1.0
W_DEBERTA = 0.35
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

# Column order of every checkpoint's head. Each checkpoint's meta.json is checked against it below.
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

# Development sample only: stop this Kaggle Python process at 600 seconds from T0.
# Hidden scoring test (>3 rows) keeps the unmodified official runtime limit.
_sample_wall_timer = None
if len(test) <= 3:
    import threading
    def _sample_wall_timeout():
        print("SAMPLE WALL LIMIT: 600 seconds reached; inference aborted", flush=True)
        os._exit(124)
    _sample_wall_timer = threading.Timer(max(0.01, 600.0 - (time.time() - T0)), _sample_wall_timeout)
    _sample_wall_timer.daemon = True
    _sample_wall_timer.start()


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))



SOURCE_CODE_SHA256 = "5864b360ef4b470726b6ae2018489e285651dfb2a305d24b1c6884666d81f800"

# Only use the own CPU-prepared artifact whose exact checksum was verified.
import hashlib, importlib.metadata, joblib
from pathlib import Path
EXPECTED_CACHE_SHA256 = "f294ce929714e852a5a3f1d34ff7a5c5f0a0df3d73bd4a24ae327cbde4d96352"
def _sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()
def _one(pattern):
    hits=sorted(set(glob.glob(f"{INPUT_DIR}/**/{pattern}",recursive=True)))
    if len(hits)!=1: raise RuntimeError(("Ambiguous or absent input",pattern,hits))
    return Path(hits[0])
mp = _one("ntl620_svc_prefit_v1/manifest.json")
manifest=json.loads(mp.read_text())
assert EXPECTED_CACHE_SHA256 != "SET_FROM_CPU_MANIFEST_BEFORE_GPU", "CPU cache checksum not pinned"
assert manifest["schema"]=="ntl620-svc-prefit-v1" and manifest["source_version"]==361311282
assert manifest["source_code_sha256"] == SOURCE_CODE_SHA256
assert manifest["labels"]==LABELS and manifest["views"]==[list(v) for v in VIEWS]
assert manifest["classes"]==list(range(K)) and not manifest["fit_used_test_rows"]
assert manifest["sample_cache_reload_exact"] is True
assert _sha(_one("train.csv"))==manifest["train_sha256"]
assert _sha(_one("pool_v4.parquet"))==manifest["pool_sha256"]
for package, expected in manifest["versions"].items():
    assert importlib.metadata.version(package)==expected, (package,expected,importlib.metadata.version(package))
cp=mp.parent/"svc_prefit.joblib"
assert manifest["cache_sha256"]==EXPECTED_CACHE_SHA256 and _sha(cp)==EXPECTED_CACHE_SHA256
cache=joblib.load(cp)  # checksum-pinned own artifact; never accept arbitrary pickle
assert cache["labels"]==LABELS and cache["svc"].classes_.tolist()==list(range(K))
B=[]
for vec,(view,lo,hi,max_features,weight) in zip(cache["vectorizers"],VIEWS):
    prep=shape_text if view=="shape" else small_words_text if view=="small" else (lambda s:s)
    B.append(vec.transform([prep(t) for t in test.text])*weight)
assert len(cache["vectorizers"])==len(VIEWS)==4
vote("svc",softmax(cache["svc"].decision_function(hstack(B,format="csr"))),W_SVC)
del B,cache
gc.collect()


In [ ]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
    del D
else:
    groups = np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)


In [ ]:
# ==============================================================================
# Mixture of Scars (MoS) 32B (Camada 32) Tri-Compass Perceptual Head
# ==============================================================================
W_32B = 0.85
W_32B_TWIN = 0.25

try:
    import torch.nn.functional as F
    from safetensors.torch import load_file
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    head_file = find("head_tricompass_32b.safetensors")
    print(f"Loading 32B MoS Tri-Compass Head from {head_file}...", flush=True)
    head_32b = load_file(head_file)

    def extract_tricompass_style_features(texts):
        feats = []
        for t in texts:
            t = str(t)
            n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
            sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]; ns = max(len(sents), 1)
            lines = [l for l in t.split(chr(10)) if l.strip()]; nl = max(len(lines), 1)
            feats.append([
                t.count(",") / n, t.count(":") / n, t.count(";") / n, t.count("—") / n, t.count(" — ") / n,
                t.count("-") / n, t.count("--") / n, t.count('"') / n, t.count("'") / n, t.count("`") / n, t.count("```") / nw,
                t.count("(") / n, t.count(")") / n, t.count("[") / n, t.count("]") / n, t.count("{") / n, t.count("}") / n,
                t.count("#") / n, t.count("*") / n, (t.count(chr(10) + "- ") + t.count(chr(10) + "* ")) / nl,
                sum(c.isdigit() for c in t) / n, sum(c.isupper() for c in t) / n,
                sum(w[0].isupper() for w in words if len(w) > 0) / nw,
                t.count(chr(10)) / nw, t.count(chr(10) + chr(10)) / nw,
                sum(len(w) for w in words) / nw, nw / ns, len(set(w.lower() for w in words)) / nw
            ])
        return np.asarray(feats, dtype=np.float32)

    style_raw = extract_tricompass_style_features(test.text.tolist())
    N_test = len(test)
    Z_32b = np.zeros((N_test, 5120), dtype=np.float32)

    # 32B MoS Layer 32 Feature Extraction
    candidates_32b = glob.glob(f"{INPUT_DIR}/**/qwen2-5-32b-instruct-bnb-4bit", recursive=True)
    if not candidates_32b:
        candidates_32b = glob.glob(f"{INPUT_DIR}/**/model-00001-of-00004.safetensors", recursive=True)

    if candidates_32b and torch.cuda.is_available():
        m_dir = candidates_32b[0] if os.path.isdir(candidates_32b[0]) else os.path.dirname(candidates_32b[0])
        print(f"Loading Qwen 32B (truncated to layer 32) from {m_dir}...", flush=True)
        tok_32b = AutoTokenizer.from_pretrained(m_dir, trust_remote_code=True)
        if tok_32b.pad_token is None:
            tok_32b.pad_token = tok_32b.eos_token
        
        cfg_32b = AutoConfig.from_pretrained(m_dir)
        cfg_32b.num_hidden_layers = 32  # MoS layer 32 truncation
        cfg_32b.use_cache = False
        
        model_32b = AutoModel.from_pretrained(
            m_dir,
            config=cfg_32b,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).eval()

        batch_size = 4
        all_reps = []
        texts = test.text.tolist()
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                b_texts = texts[i:i + batch_size]
                inp = tok_32b(b_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda:0")
                out = model_32b(**inp)
                mask = inp.attention_mask.unsqueeze(-1).float()
                # Mean pooling at layer 32
                pooled = (out.last_hidden_state.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                all_reps.append(pooled.cpu().numpy())
        Z_32b = np.concatenate(all_reps, axis=0)
        del model_32b
        torch.cuda.empty_cache()
        gc.collect()
        print(f"✓ 32B MoS Layer 32 features extracted successfully: {Z_32b.shape}", flush=True)

    # Combine 32B features with stylometrics (5120 + 23 = 5143)
    X_32b_raw = np.hstack([Z_32b, style_raw])
    mean_vec = head_32b["scaler.mean"].float().numpy()
    scale_vec = head_32b["scaler.scale"].float().numpy()
    weight_mat = head_32b["head.weight"].float().numpy()
    bias_vec = head_32b["head.bias"].float().numpy()

    X_normed = (X_32b_raw - mean_vec) / scale_vec
    logits_32b = X_normed @ weight_mat.T + bias_vec
    probs_32b = softmax(logits_32b)
    vote("qwen32b_mos_tricompass", probs_32b, W_32B)

    # Prompt Centering Twin Inference
    if twin_rows.any():
        Zc = Z_32b.copy()
        Sc = style_raw.copy()
        for g in np.unique(groups[twin_rows]):
            m = (groups == g)
            if m.sum() > 1:
                Zc[m] -= Zc[m].mean(0)
                Sc[m] -= Sc[m].mean(0)
        X_c_raw = np.hstack([Zc, Sc])
        X_c = (X_c_raw - mean_vec) / scale_vec
        logits_c = X_c[twin_rows] @ weight_mat.T + bias_vec
        vote("qwen32b_mos_twin", softmax(logits_c), W_32B_TWIN, rows=twin_rows)

    print("✓ 32B MoS (Layer 32) Tri-Compass inference recorded!", flush=True)

except Exception as e:
    print(f"Warning: 32B MoS head evaluation bypassed: {e}", flush=True)
    import traceback
    traceback.print_exc()


In [ ]:
def sinkhorn(P, iters=300):
    """One scale per class, so that every class gets an equal share of the probability. Rows are renormalised."""
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)


assert np.isfinite(LOGIT).all(), "Nonfinite ensemble logits"
P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3 row public test.csv
    P = sinkhorn(P)
assert np.isfinite(P).all(), "Nonfinite ensemble probability"
assert P.shape == (len(test), K), "Probability shape mismatch"
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")

if _sample_wall_timer is not None:
    _sample_wall_timer.cancel()
